In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from skimage.io import imread
from skimage.transform import resize
import numpy as np
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu, sobel
from skimage.transform import resize
from skimage.feature import canny
from skimage import morphology, measure
import numpy as np
import torch
from skimage import measure, morphology

from skimage import measure, morphology
import numpy as np

def count_holes(binary_image):
    """
    binary_image: 2D numpy array, foreground=1, background=0
    returns: number of holes in the foreground object
    """
    # Label foreground objects
    labeled_foreground = measure.label(binary_image)
    num_holes_total = 0

    for region in measure.regionprops(labeled_foreground):
        # Extract mask of this object
        mask = labeled_foreground == region.label
        # Invert mask: inside object becomes 0, outside 1
        inv_mask = np.logical_not(mask)
        # Label connected background regions inside object
        # Only consider regions fully inside the object (exclude external background)
        labeled_bg = measure.label(inv_mask, connectivity=1)
        # Count background regions that are fully inside the object
        # We can subtract 1 if the outer background is labeled as a region touching the border
        # But simplest: mask border pixels and count connected components inside
        # First, remove background touching the border
        border_mask = np.copy(labeled_bg)
        border_labels = np.unique(np.concatenate([
            border_mask[0, :], border_mask[-1, :], border_mask[:, 0], border_mask[:, -1]
        ]))
        for lbl in border_labels:
            border_mask[border_mask == lbl] = 0
        # Remaining nonzero labels are holes
        num_holes = len(np.unique(border_mask)) - 1  # subtract 0
        num_holes_total += num_holes

    return num_holes_total



class SkimageDataset(Dataset):
    def __init__(self, root_dir, image_size=(128, 128), transform=None, binary=False):
        self.samples = []
        self.class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.class_names)}
        self.image_size = image_size
        self.transform = transform
        self.binary = binary   # <-- this was missing!

        for cls_name in self.class_names:
            cls_path = os.path.join(root_dir, cls_name)
            if not os.path.isdir(cls_path):
                continue
            for fname in os.listdir(cls_path):
                if fname.lower().endswith((".jpg", ".png", ".jpeg")):
                    self.samples.append((os.path.join(cls_path, fname), self.class_to_idx[cls_name]))

    def __len__(self):
        # Return number of samples
        return len(self.samples)


    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = imread(path)
        img = resize(img, self.image_size, anti_aliasing=False)

        if self.binary:
            # 1. Convert to grayscale
            img_gray = rgb2gray(img)

            # 2. Otsu threshold on intensity (like Fiji "Make Binary")
            thresh = threshold_otsu(img_gray)
            img_binary = (img_gray > thresh).astype(np.float32)

            # 3. Invert (Fiji usually makes foreground=white, background=black)
            img_binary = 1.0 - img_binary
            holes = count_holes(img_binary)     
            features = []
            labeled_foreground = measure.label(img_binary)
            for region in measure.regionprops(labeled_foreground):
                features.append([
                    region.area,
                    region.perimeter,
                    region.eccentricity,
                    holes
                ])

            # 4. Convert to tensor
            img = torch.tensor(img_binary, dtype=torch.float32).unsqueeze(0)

        else:
            if self.transform:
                img = self.transform(img)
            img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)

        label = torch.tensor(label, dtype=torch.long)
        return (img, features), label

In [2]:
import matplotlib.pyplot as plt

def show_batch_images_matplotlib(batch_imgs, batch_labels):
    """
    batch_imgs: Tensor of shape (B, C, H, W)
    batch_labels: Tensor of shape (B,)
    """
    batch_size = batch_imgs.shape[0]
    plt.figure(figsize=(15, 5))
    
    for i in range(batch_size):
        img = batch_imgs[i]
        label = batch_labels[i]
        
        if img.shape[0] == 1:  # grayscale
            img_to_show = img.squeeze(0).numpy()
            cmap = 'gray'
        else:  # RGB
            img_to_show = img.permute(1, 2, 0).numpy()
            cmap = None
        
        plt.subplot(1, batch_size, i+1)
        plt.imshow(img_to_show, cmap=cmap)
        plt.title(f"Label: {label.item()}")
        plt.axis('off')
        print()
    
    plt.show()



In [3]:
dataset = SkimageDataset("./splited_dataset/train", image_size=(128, 128), binary=True)

In [4]:

# dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# # Usage with your DataLoader
# for batch_imgs, batch_labels in dataloader:
#     print(batch_imgs.shape)   # (16, 3, 128, 128)
#     print(batch_labels.shape) # (16,)
#     show_batch_images_matplotlib(batch_imgs, batch_labels)
#     break

In [5]:
for features, labels in dataset: 
    print(features[0])
    print(features[1])
    break

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]])
[[np.float64(2242.0), np.float64(283.07716446627535), 0.9116188011938468, 0], [np.float64(432.0), np.float64(130.69238815542514), 0.9142523644939617, 0], [np.float64(1091.0), np.float64(224.00609665440987), 0.86108816447601, 0], [np.float64(872.0), np.float64(170.00609665440987), 0.8590816687351112, 0]]


In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from collections import defaultdict

# Prepare the data
X = []
y = []

for features, label in dataset:
    X.append(features[1])  
    y.append(label.item())


X = np.array(X)
y = np.array(y)
x = X
y = y
print(X.shape, y.shape)  # (num_samples, 16384), (num_samples,)


In [ ]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x,y, test_size=0.2)

In [ ]:
# 2️⃣ List of classifiers
classifiers = [
    ("SVM", SVC()),
    ("Random Forest", RandomForestClassifier()),
    ("Logistic Regression", LogisticRegression(max_iter=1000))
]

# 3️⃣ Run each classifier and save results
results = []

for name, clf in classifiers:
    scores = cross_val_score(clf, xtrain, ytrain, cv=10)  # 5-fold CV
    mean_acc = scores.mean()
    results.append({"Classifier": name, "Mean Accuracy": mean_acc})

In [ ]:
import pandas as pd
# 4️⃣ Convert results to DataFrame
df_results = pd.DataFrame(results)
print(df_results)